In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models

load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Carregamos as variáveis de ambiente do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada da variável de ambiente
)

# Configura modelo padrão para todas operações DSPy
dspy.configure(lm=lm)

In [3]:
class ResponderPergunta(dspy.Signature):
    """Responda pergunta de forma correta, clara e didática."""

    # Campo entrada: pergunta a ser respondida
    pergunta: str = dspy.InputField(
        desc="Pergunta que deve ser respondida"
    )

    # Campo saída: resposta completa
    resposta: str = dspy.OutputField(
        desc="Resposta completa e didática"
    )

In [4]:
# O programa que será executado N vezes pelo BestOfN
programa_base = dspy.Predict(ResponderPergunta)

In [5]:
class AvaliarResposta(dspy.Signature):
    """
    Avalie rigorosamente uma solução proposta para o problema
    de processamento distribuído de pagamentos.

    A resposta deve ser avaliada considerando:

    1. Correção técnica.
    2. Garantia contra cobranças duplicadas.
    3. Tratamento correto de idempotência.
    4. Tratamento de concorrência e race conditions.
    5. Tratamento de retries e timeouts.
    6. Tratamento do caso em que o gateway processou a cobrança,
       mas a resposta foi perdida.
    7. Estratégia de persistência e estados da transação.
    8. Estratégia de recuperação após falhas.
    9. Consistência entre banco de dados e sistema externo.
    10. Discussão correta de exactly-once e at-least-once.
    11. Uso apropriado de filas/mensageria quando necessário.
    12. Discussão dos trade-offs da arquitetura.
    13. Clareza do fluxo completo da operação.
    14. Qualidade dos cenários de falha apresentados.

    Seja rigoroso.

    Não atribua nota acima de 0.90 se algum dos aspectos críticos
    estiver ausente ou tecnicamente incorreto.

    Notas:

    0.0-0.3:
        Solução incorreta ou superficial.

    0.4-0.6:
        Solução parcialmente correta, mas com lacunas importantes.

    0.7-0.8:
        Boa solução, porém faltam alguns detalhes relevantes.

    0.8-0.9:
        Solução muito boa e tecnicamente consistente.

    0.9-1.0:
        Solução excepcional, completa, tecnicamente rigorosa
        e que aborda corretamente os principais modos de falha.
    """

    pergunta: str = dspy.InputField(
        desc="Problema de arquitetura apresentado"
    )

    resposta: str = dspy.InputField(
        desc="Solução proposta que deve ser avaliada"
    )

    nota: float = dspy.OutputField(
        desc="Nota rigorosa entre 0.0 e 1.0"
    )

    justificativa: str = dspy.OutputField(
        desc="Justificativa detalhada apontando qualidades e deficiências"
    )

In [6]:
# Avaliador que BestOfN usará para pontuar respostas
# DSPy utilizará LM configurado globalmente
juiz = dspy.Predict(AvaliarResposta)

In [7]:
# Armazena dados de cada avaliação para análise posterior
avaliacoes = []

In [8]:
def avaliar_resposta(args, pred):
    """Avalia resposta e retorna score para BestOfN."""

    # Chama avaliador para pontuar a resposta
    avaliacao = juiz(
        pergunta=args["pergunta"],
        resposta=pred.resposta,
    )

    # Normaliza nota entre 0 e 1 (segurança)
    nota = max(0.0, min(1.0, float(avaliacao.nota)))

    # Armazena para análise posterior
    avaliacoes.append({
        "resposta": pred.resposta,
        "nota": nota,
        "justificativa": avaliacao.justificativa,
    })

    return nota  # BestOfN usa isso para selecionar melhor resposta

## Criando BestOfN

Cria módulo que executará programa_base 3 vezes e selecionará resposta com maior score.

Parâmetros:
- `module`: programa a executar N vezes
- `N`: número de tentativas
- `reward_fn`: função que retorna score (0-1)
- `threshold`: score mínimo para aceitar resposta (99% de qualidade)

In [9]:
# Módulo BestOfN que gerará 3 respostas e escolherá melhor
best_of_n = dspy.BestOfN(
    module=programa_base,      # Módulo a executar
    N=3,                        # Gerar 3 respostas
    reward_fn=avaliar_resposta, # Função de avaliação
    threshold=0.95,             # Aceitar se score >= 0.95
)

In [10]:
pergunta = """
Você precisa projetar um sistema de processamento de pagamentos distribuído.

O sistema recebe requisições HTTP para cobrar clientes e possui os seguintes requisitos:

1. Uma mesma requisição pode ser enviada várias vezes devido a retries do cliente.
2. O serviço de pagamento externo pode processar a cobrança com sucesso, mas a resposta pode ser perdida por timeout.
3. Existem múltiplas instâncias da aplicação processando requisições simultaneamente.
4. O banco de dados pode sofrer falhas temporárias.
5. O sistema não pode cobrar o cliente duas vezes pela mesma operação.
6. O throughput esperado é de milhares de transações por segundo.

Explique como você projetaria esse sistema.

Sua resposta deve abordar obrigatoriamente:

- idempotência;
- geração e armazenamento de idempotency keys;
- concorrência entre múltiplas instâncias;
- transações no banco de dados;
- estados intermediários da operação;
- retries;
- timeouts;
- falhas do serviço de pagamento;
- o problema de uma cobrança ter sido realizada externamente,
  mas a aplicação não ter recebido a confirmação;
- consistência entre banco de dados e serviço externo;
- possíveis usos de filas ou mensageria;
- exatamente-once versus at-least-once;
- como evitar race conditions;
- estratégia de recuperação após falhas;
- principais trade-offs da solução.

Apresente uma arquitetura proposta e explique passo a passo o fluxo de uma cobrança,
incluindo pelo menos dois cenários de falha e como o sistema se recuperaria deles.
"""

In [11]:
# Executa BestOfN: gera 3 respostas, avalia cada uma, retorna melhor
resultado = best_of_n(
    pergunta=pergunta
)

In [12]:
print("MELHOR RESPOSTA")
print("=" * 70)
print(resultado.resposta)

MELHOR RESPOSTA
Resumo da solução proposta (visão geral)
- Objetivo: garantir que cada cobrança seja executada efetivamente uma vez (evitar dupla cobrança), suportar retries, alta concorrência (milhares TPS), tolerar falhas temporárias no BD e no serviço de pagamento externo.
- Estratégia principal: usar idempotency keys, um registro persistente do pedido de cobrança com estados bem definidos, única influência atômica no banco de dados (constraints + transações/compare-and-set), processamento assíncrono desacoplado via fila, outbox pattern para consistência entre DB e mensagens, e jobs de reconciliação para casos em que a confirmação do pagamento foi perdida.

Componentes principais da arquitetura
1. API Gateway / Frontend HTTP (stateless)
2. Serviço de pagamentos (stateless, várias instâncias)
3. Banco de dados transacional (tabela de pagamentos / idempotency)
4. Fila/mensagem de trabalho (Kafka / Pulsar / RabbitMQ) — alta taxa: Kafka/Pulsar preferível
5. Workers consumidores que exec

## Analisar Todas Tentativas

Mostra as 3 respostas geradas, suas notas e justificativas. Permite ver razão da seleção.

In [13]:
# Itera sobre histórico de avaliações guardadas
for i, avaliacao in enumerate(avaliacoes, start=1):

    print(f"\n{'=' * 70}")
    print(f"TENTATIVA {i}")
    print(f"{'=' * 70}")

    print("\nRESPOSTA:")
    print(avaliacao["resposta"])

    print("\nNOTA:")
    print(avaliacao["nota"])

    print("\nJUSTIFICATIVA:")
    print(avaliacao["justificativa"])


TENTATIVA 1

RESPOSTA:
Resumo da solução proposta (visão geral)
- Objetivo: garantir que cada cobrança seja executada efetivamente uma vez (evitar dupla cobrança), suportar retries, alta concorrência (milhares TPS), tolerar falhas temporárias no BD e no serviço de pagamento externo.
- Estratégia principal: usar idempotency keys, um registro persistente do pedido de cobrança com estados bem definidos, única influência atômica no banco de dados (constraints + transações/compare-and-set), processamento assíncrono desacoplado via fila, outbox pattern para consistência entre DB e mensagens, e jobs de reconciliação para casos em que a confirmação do pagamento foi perdida.

Componentes principais da arquitetura
1. API Gateway / Frontend HTTP (stateless)
2. Serviço de pagamentos (stateless, várias instâncias)
3. Banco de dados transacional (tabela de pagamentos / idempotency)
4. Fila/mensagem de trabalho (Kafka / Pulsar / RabbitMQ) — alta taxa: Kafka/Pulsar preferível
5. Workers consumidores 